In [1]:
import json
import pickle
from pathlib import Path

import pandas as pd
from PIL import Image
from io import BytesIO

PARQUET_PATH = Path("data/mmstar.parquet")
IDS_PATH = Path("data/question_ids/mmstar_ids.json")
OUT_PATH = Path("data/mmstar_subset.pkl")

# Quick-check subset: set to None to keep all 712 ids,
# or to an int (e.g. 10) to dump only the first N samples to a separate file.
SAMPLE_LIMIT = 10
SMALL_OUT_PATH = Path("data/mmstar_subset_small.pkl")

df = pd.read_parquet(PARQUET_PATH)
print("parquet rows :", len(df))
print("columns      :", df.columns.tolist())

sample = df.iloc[0]
print("\nfirst row 'index'   :", sample["index"])
print("first row 'question':", sample["question"][:120], "...")
print("type(row['image'])  :", type(sample["image"]))
if isinstance(sample["image"], dict):
    print("image keys          :", list(sample["image"].keys()))
    Image.open(BytesIO(sample["image"]["bytes"])).convert("RGB").resize((128, 128))

parquet rows : 1500
columns      : ['index', 'question', 'answer', 'category', 'l2_category', 'image', 'meta_info']

first row 'index'   : 0
first row 'question': Which option describe the object relationship in the image correctly?
Options: A: The suitcase is on the book., B: The s ...
type(row['image'])  : <class 'bytes'>


In [2]:
with open(IDS_PATH) as f:
    keep_ids = json.load(f)
print("ids in mmstar_ids.json:", len(keep_ids))

subset = df[df["index"].isin(keep_ids)].reset_index(drop=True)
print("subset rows           :", len(subset))

missing = sorted(set(keep_ids) - set(subset["index"].tolist()))
if missing:
    print(f"warning: {len(missing)} ids not found in parquet, e.g. {missing[:10]}")

records = subset.to_dict(orient="records")

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, "wb") as f:
    pickle.dump(records, f)

print(f"saved {len(records)} records -> {OUT_PATH}")

if SAMPLE_LIMIT is not None:
    small_records = records[:SAMPLE_LIMIT]
    SMALL_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(SMALL_OUT_PATH, "wb") as f:
        pickle.dump(small_records, f)
    print(f"saved {len(small_records)} records -> {SMALL_OUT_PATH}")

ids in mmstar_ids.json: 712
subset rows           : 712
saved 712 records -> data/mmstar_subset.pkl
saved 10 records -> data/mmstar_subset_small.pkl


In [1]:
import pandas as pd

In [3]:
df = pd.read_parquet('data/mmstar.parquet')

In [5]:
df['image'].values[0]

b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x08\x06\x06\x07\x06\x05\x08\x07\x07\x07\t\t\x08\n\x0c\x14\r\x0c\x0b\x0b\x0c\x19\x12\x13\x0f\x14\x1d\x1a\x1f\x1e\x1d\x1a\x1c\x1c $.\' ",#\x1c\x1c(7),01444\x1f\'9=82<.342\xff\xdb\x00C\x01\t\t\t\x0c\x0b\x0c\x18\r\r\x182!\x1c!22222222222222222222222222222222222222222222222222\xff\xc0\x00\x11\x08\x01\x80\x02\x00\x03\x01"\x00\x02\x11\x01\x03\x11\x01\xff\xc4\x00\x1f\x00\x00\x01\x05\x01\x01\x01\x01\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x01\x02\x03\x04\x05\x06\x07\x08\t\n\x0b\xff\xc4\x00\xb5\x10\x00\x02\x01\x03\x03\x02\x04\x03\x05\x05\x04\x04\x00\x00\x01}\x01\x02\x03\x00\x04\x11\x05\x12!1A\x06\x13Qa\x07"q\x142\x81\x91\xa1\x08#B\xb1\xc1\x15R\xd1\xf0$3br\x82\t\n\x16\x17\x18\x19\x1a%&\'()*456789:CDEFGHIJSTUVWXYZcdefghijstuvwxyz\x83\x84\x85\x86\x87\x88\x89\x8a\x92\x93\x94\x95\x96\x97\x98\x99\x9a\xa2\xa3\xa4\xa5\xa6\xa7\xa8\xa9\xaa\xb2\xb3\xb4\xb5\xb6\xb7\xb8\xb9\xba\xc2\xc3\xc4\xc5\xc6\xc7\xc8\xc9\xca\xd2\xd3\

In [6]:
df.columns

Index(['index', 'question', 'answer', 'category', 'l2_category', 'image',
       'meta_info'],
      dtype='object')

In [9]:
data = pd.read_parquet('data/mmstar.parquet')

In [13]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   index        1500 non-null   int64 
 1   question     1500 non-null   object
 2   answer       1500 non-null   object
 3   category     1500 non-null   object
 4   l2_category  1500 non-null   object
 5   image        1500 non-null   object
 6   meta_info    1500 non-null   object
dtypes: int64(1), object(6)
memory usage: 82.2+ KB
